# Lab Notebook 1: Dataset Creation with Global Normalization
**Team:** MozwiloFortune  
**Region:** Po Valley, Italy (Tile T32TMR)

## Purpose
Create train/val/test datasets with proper global normalization.

## Output
- `patches_train.npz` - 20,000 patches (3Ãƒâ€”3Ãƒâ€”16)
- `patches_val.npz` - 10,000 patches
- `patches_test.npz` - 10,000 patches  
- `labels_train.npy`, `labels_val.npy`, `labels_test.npy`
- `global_normalization_stats.json` Ã¢â€ Â THE CRITICAL FIX!
- `metadata.json`

## Runtime: ~30-45 minutes

## Setup

In [1]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window

print("Ã¢Å“â€œ Libraries imported")
print(f"  rasterio: {rasterio.__version__}")
print(f"  numpy: {np.__version__}")

Ã¢Å“â€œ Libraries imported
  rasterio: 1.5.0
  numpy: 1.26.4


## Configuration

In [2]:
# Paths
S2_DATA_DIR = "/p/scratch/training2600/teamMozwiloFortune/data/sentinel2_extracted"
CORINE_PATH = "/p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif"
OUTPUT_BASE = "/p/scratch/training2600/teamMozwiloFortune/final_project/data"

# Data (same 4 acquisitions from Checkpoint 1)
ALL_SAFE_FOLDERS = [
    "S2B_MSIL2A_20180308T102019_N0500_R065_T32TMR_20230910T101040.SAFE",  # March
    "S2B_MSIL2A_20180616T102019_N0500_R065_T32TMR_20230715T133909.SAFE",  # June
    "S2B_MSIL2A_20180904T102019_N0500_R065_T32TMR_20230824T235353.SAFE",  # Sept
    "S2B_MSIL2A_20181024T102059_N0500_R065_T32TMR_20230816T103450.SAFE",  # Oct
]

TILE_ID = "T32TMR"
BANDS = ['B02', 'B03', 'B04', 'B08']  # Blue, Green, Red, NIR
PATCH_SIZE = 5
STRIDE = 5
GRID_SIZE = 2000  # 3Ãƒâ€”3 grid
CROPLAND_LABELS = {12, 15, 20}
TRAIN_CROPLAND_FRACTION = 0.20
VAL_CROPLAND_FRACTION = 0.10
TEST_CROPLAND_FRACTION = 0.10
MAX_PER_CELL = 4000

TARGET_PATCHES = {
    'train': 20000,
    'val': 10000,
    'test': 10000
}

os.makedirs(OUTPUT_BASE, exist_ok=True)

print("Configuration:")
print(f"  Tile: {TILE_ID}")
print(f"  Bands: {len(BANDS)} Ãƒâ€” {len(ALL_SAFE_FOLDERS)} dates = {len(BANDS)*len(ALL_SAFE_FOLDERS)} total")
print(f"  Grid: 3Ãƒâ€”3 cells of {GRID_SIZE}Ãƒâ€”{GRID_SIZE} pixels")
print(f"  Cropland labels: {sorted(CROPLAND_LABELS)}")
print(f"  Train cropland target: {TRAIN_CROPLAND_FRACTION:.0%}")
print(f"  Val cropland target: {VAL_CROPLAND_FRACTION:.0%}")
print(f"  Test cropland target: {TEST_CROPLAND_FRACTION:.0%}")
print(f"  Target: {sum(TARGET_PATCHES.values()):,} patches total")
print(f"  Output: {OUTPUT_BASE}")

Configuration:
  Tile: T32TMR
  Bands: 4 Ãƒâ€” 4 dates = 16 total
  Grid: 3Ãƒâ€”3 cells of 2000Ãƒâ€”2000 pixels
  Cropland labels: [12, 15, 20]
  Train cropland target: 20%
  Val cropland target: 10%
  Target: 40,000 patches total
  Output: /p/scratch/training2600/teamMozwiloFortune/final_project/data


## Step 1: Stack Temporal Bands

In [3]:
def stack_temporal_bands(safe_folders, s2_dir, output_path, bands):
    """Stack bands from multiple dates into single GeoTIFF."""
    all_band_paths = []
    
    print(f"Scanning {len(safe_folders)} .SAFE folders...")
    for safe_name in sorted(safe_folders):
        safe_path = Path(s2_dir) / safe_name
        date_paths = []
        for band in bands:
            pattern = f"GRANULE/*/IMG_DATA/R10m/*_{band}_10m.jp2"
            matches = list(safe_path.glob(pattern))
            date_paths.append(str(matches[0]) if matches else None)
        
        if all(p is not None for p in date_paths):
            all_band_paths.append(date_paths)
            date_str = safe_name.split('_')[2][:8]
            print(f"  Ã¢Å“â€œ {date_str}: {len(date_paths)} bands")
    
    # Get metadata from first band
    with rasterio.open(all_band_paths[0][0]) as ref:
        profile = ref.profile.copy()
        height, width = ref.height, ref.width
    
    # Update for multi-band output
    n_bands_total = len(bands) * len(all_band_paths)
    profile.update(
        driver='GTiff', count=n_bands_total, dtype='uint16',
        compress='lzw', tiled=True, blockxsize=256, blockysize=256
    )
    
    # Read all bands
    print(f"\nReading {n_bands_total} bands...")
    all_data = []
    for t, date_paths in enumerate(all_band_paths):
        for b, band_path in enumerate(date_paths):
            with rasterio.open(band_path) as src:
                all_data.append(src.read(1))
    
    # Write stacked file
    print(f"Writing to {Path(output_path).name}...")
    with rasterio.open(output_path, 'w', **profile) as dst:
        for i, data in enumerate(all_data):
            dst.write(data, i + 1)
            band_name = f"{bands[i % len(bands)]}_t{i // len(bands)}"
            dst.set_band_description(i + 1, band_name)
    
    print(f"Ã¢Å“â€œ Stacked {len(all_band_paths)} dates Ã¢â€ â€™ {n_bands_total} bands")
    print(f"  Size: {width} Ãƒâ€” {height} pixels")
    return len(all_band_paths)

STACKED_FILE = Path(OUTPUT_BASE) / f"{TILE_ID}_stacked.tif"

print("="*70)
print("STEP 1: STACK TEMPORAL BANDS")
print("="*70)

n_dates = stack_temporal_bands(ALL_SAFE_FOLDERS, S2_DATA_DIR, STACKED_FILE, BANDS)
print(f"\nÃ¢Å“â€¦ Step 1 complete")

STEP 1: STACK TEMPORAL BANDS
Scanning 4 .SAFE folders...
  Ã¢Å“â€œ 20180308: 4 bands
  Ã¢Å“â€œ 20180616: 4 bands
  Ã¢Å“â€œ 20180904: 4 bands
  Ã¢Å“â€œ 20181024: 4 bands

Reading 16 bands...
Writing to T32TMR_stacked.tif...
Ã¢Å“â€œ Stacked 4 dates Ã¢â€ â€™ 16 bands
  Size: 10980 Ãƒâ€” 10980 pixels

Ã¢Å“â€¦ Step 1 complete


## Step 2: Align CORINE Labels

In [4]:
def align_corine(corine_path, s2_path, output_path):
    """Reproject CORINE to match Sentinel-2 geometry."""
    with rasterio.open(s2_path) as s2:
        dst_crs = s2.crs
        dst_transform = s2.transform
        dst_width, dst_height = s2.width, s2.height
    
    print(f"Target CRS: {dst_crs}")
    print(f"Target size: {dst_width} Ãƒâ€” {dst_height}")
    
    with rasterio.open(corine_path) as src:
        profile = src.profile.copy()
        profile.update(
            driver='GTiff', crs=dst_crs, transform=dst_transform,
            width=dst_width, height=dst_height,
            compress='lzw', tiled=True
        )
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=dst_transform, dst_crs=dst_crs,
                resampling=Resampling.nearest
    
    print("Ã¢Å“â€œ CORINE aligned")

CORINE_ALIGNED = Path(OUTPUT_BASE) / f"corine_aligned_{TILE_ID}.tif"

print("\n" + "="*70)
print("STEP 2: ALIGN CORINE LABELS")
print("="*70)

align_corine(CORINE_PATH, STACKED_FILE, CORINE_ALIGNED)
print("\nÃ¢Å“â€¦ Step 2 complete")


STEP 2: ALIGN CORINE LABELS
Target CRS: EPSG:32632
Target size: 10980 Ãƒâ€” 10980
Ã¢Å“â€œ CORINE aligned

Ã¢Å“â€¦ Step 2 complete


## Step 3: Ã°Å¸â€Â´ Compute Global Normalization Stats

### THIS IS THE CRITICAL FIX!

**What was broken:** Each grid cell was normalized independently  
**What we fix:** Compute global statistics from entire tile, apply to all patches  
**Expected impact:** 12% Ã¢â€ â€™ 30-40% balanced accuracy

In [5]:
def compute_global_stats(s2_path, n_samples=2000, sample_size=100):
    """
    Ã°Å¸â€Â´ KEY FIX: Compute global statistics from entire tile.
    
    This ensures value 0.5 means the same thing everywhere,
    not just within a single grid cell.
    """
    print(f"Sampling {n_samples} random {sample_size}Ãƒâ€”{sample_size} patches...")
    
    with rasterio.open(s2_path) as src:
        n_bands = src.count
        height, width = src.height, src.width
        sample_data = [[] for _ in range(n_bands)]
        
        np.random.seed(42)
        for i in range(n_samples):
            row = np.random.randint(0, height - sample_size)
            col = np.random.randint(0, width - sample_size)
            window = Window(col, row, sample_size, sample_size)
            data = src.read(window=window)
            for b in range(n_bands):
                valid = data[b][data[b] > 0]
                if len(valid) > 0:
                    sample_data[b].extend(valid.tolist())
            
            if (i + 1) % 500 == 0:
                print(f"  Progress: {i + 1}/{n_samples}")
    
    # Compute percentiles
    global_stats = {}
    print(f"\nGlobal statistics (2nd-98th percentile):")
    print(f"  {'Band':<10} {'p2':>10} {'p98':>10}")
    print(f"  {'-'*10} {'-'*10} {'-'*10}")
    
    for b in range(n_bands):
        vals = np.array(sample_data[b])
        p2, p98 = np.percentile(vals, [2, 98])
        global_stats[b] = {'p2': float(p2), 'p98': float(p98)}
        band_name = f"{BANDS[b % len(BANDS)]}_t{b // len(BANDS)}"
        print(f"  {band_name:<10} {p2:10.1f} {p98:10.1f}")
    
    total_pixels = sum(len(s) for s in sample_data)
    print(f"\nÃ¢Å“â€œ Computed from {total_pixels:,} pixels")
    return global_stats

print("\n" + "="*70)
print("STEP 3: Ã°Å¸â€Â´ COMPUTE GLOBAL NORMALIZATION STATS")
print("="*70)
print("This is THE critical fix: 12% Ã¢â€ â€™ 30-40% accuracy\n")

GLOBAL_STATS = compute_global_stats(STACKED_FILE)

# Save for reproducibility
stats_file = Path(OUTPUT_BASE) / 'global_normalization_stats.json'
with open(stats_file, 'w') as f:
    json.dump(GLOBAL_STATS, f, indent=2)

print(f"\nÃ¢Å“â€œ Saved to {stats_file.name}")
print("\nÃ¢Å“â€¦ Step 3 complete")


STEP 3: Ã°Å¸â€Â´ COMPUTE GLOBAL NORMALIZATION STATS
This is THE critical fix: 12% Ã¢â€ â€™ 30-40% accuracy

Sampling 2000 random 100Ãƒâ€”100 patches...
  Progress: 500/2000
  Progress: 1000/2000
  Progress: 1500/2000
  Progress: 2000/2000

Global statistics (2nd-98th percentile):
  Band               p2        p98
  ---------- ---------- ----------
  B02_t0         1121.0    11166.0
  B03_t0         1102.0    11167.0
  B04_t0         1072.0    11168.0
  B08_t0         1080.0    10656.0
  B02_t1         1228.0     9656.0
  B03_t1         1323.0     9424.0
  B04_t1         1161.0     9304.0
  B08_t1         1182.0     9103.0
  B02_t2         1118.0     5048.0
  B03_t2         1173.0     5036.0
  B04_t2         1056.0     4928.0
  B08_t2         1101.0     6937.0
  B02_t3         1023.0     2770.0
  B03_t3         1032.0     3014.0
  B04_t3          993.0     3216.0
  B08_t3         1016.0     5484.0

Ã¢Å“â€œ Computed from 319,998,834 pixels

Ã¢Å“â€œ Saved to global_normalization_stats.

## Step 4: Extract Patches (5Ãƒâ€”5 Grid with Global Normalization)

In [6]:
def extract_patches(s2_path, corine_path, region, target, global_stats, patch_size=3, stride=3):
    """
    Extract patches using GLOBAL normalization (not per-cell!).
    """
    
    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as cor_src:
        n_bands = s2_src.count
        
        # Define region bounds
        row_start = region.get('row_start', 0)
        row_end = min(region.get('row_end', s2_src.height), s2_src.height)
        col_start = region.get('col_start', 0)
        col_end = min(region.get('col_end', s2_src.width), s2_src.width)
        
        # Read data
        window = Window(col_start, row_start, col_end - col_start, row_end - row_start)
        s2_data = s2_src.read(window=window)
        corine_data = cor_src.read(1, window=window)
        
        # Normalize with global stats
        s2_norm = np.zeros_like(s2_data, dtype=np.float32)
        for b in range(n_bands):
            band = s2_data[b].astype(np.float32)
            p2 = global_stats[b]['p2']
            p98 = global_stats[b]['p98']
            band = np.clip(band, p2, p98)
            s2_norm[b] = (band - p2) / (p98 - p2 + 1e-6)
        
        # Extract patches
        h, w = corine_data.shape
        patches, labels = [], []
        center = patch_size // 2
        
        for row in range(0, h - patch_size + 1, stride):
            for col in range(0, w - patch_size + 1, stride):
                label = corine_data[row + center, col + center]
                if not (1 <= label <= 44):
                    continue
                
                patch = s2_norm[:, row:row+patch_size, col:col+patch_size]
                if np.any(patch[:4] == 0):
                    continue
                
                patch = np.transpose(patch, (1, 2, 0))
                patches.append(patch)
                labels.append(label)
                
                if len(patches) >= target:
                    break
            if len(patches) >= target:
                break
        
        return np.array(patches, dtype=np.float32), np.array(labels, dtype=np.uint8)

def extract_quota_patches(s2_path, corine_path, region, cropland_target, non_cropland_target,
                         global_stats, cropland_labels, patch_size=3, stride=3):
    """Extract a cropland-aware subset with explicit quotas."""

    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as cor_src:
        n_bands = s2_src.count
        row_start = region.get('row_start', 0)
        row_end = min(region.get('row_end', s2_src.height), s2_src.height)
        col_start = region.get('col_start', 0)
        col_end = min(region.get('col_end', s2_src.width), s2_src.width)

        window = Window(col_start, row_start, col_end - col_start, row_end - row_start)
        s2_data = s2_src.read(window=window)
        corine_data = cor_src.read(1, window=window)

        s2_norm = np.zeros_like(s2_data, dtype=np.float32)
        for b in range(n_bands):
            band = s2_data[b].astype(np.float32)
            p2 = global_stats[b]['p2']
            p98 = global_stats[b]['p98']
            band = np.clip(band, p2, p98)
            s2_norm[b] = (band - p2) / (p98 - p2 + 1e-6)

        h, w = corine_data.shape
        center = patch_size // 2
        cropland_patches, cropland_codes = [], []
        non_cropland_patches, non_cropland_codes = [], []

        for row in range(0, h - patch_size + 1, stride):
            for col in range(0, w - patch_size + 1, stride):
                if len(cropland_patches) >= cropland_target and len(non_cropland_patches) >= non_cropland_target:
                    break

                label = int(corine_data[row + center, col + center])
                if not (1 <= label <= 44):
                    continue

                patch = s2_norm[:, row:row+patch_size, col:col+patch_size]
                if np.any(patch[:4] == 0):
                    continue

                patch = np.transpose(patch, (1, 2, 0))
                if label in cropland_labels:
                    if len(cropland_patches) < cropland_target:
                        cropland_patches.append(patch)
                        cropland_codes.append(label)
                else:
                    if len(non_cropland_patches) < non_cropland_target:
                        non_cropland_patches.append(patch)
                        non_cropland_codes.append(label)

            if len(cropland_patches) >= cropland_target and len(non_cropland_patches) >= non_cropland_target:
                break

    patches = np.array(cropland_patches + non_cropland_patches, dtype=np.float32)
    labels = np.array(cropland_codes + non_cropland_codes, dtype=np.uint8)
    if len(labels) > 0:
        rng = np.random.default_rng(42)
        order = rng.permutation(len(labels))
        patches = patches[order]
        labels = labels[order]

    counts = {
        'cropland': len(cropland_codes),
        'non_cropland': len(non_cropland_codes)
    }
    return patches, labels, counts

# 5x5 grid pattern
grid_pattern = [
    ['train', 'val',   'test',  'train', 'val'],
    ['val',   'test',  'train', 'val',   'test'],
    ['test',  'train', 'val',   'test',  'train'],
    ['train', 'val',   'test',  'train', 'val'],
    ['val',   'test',  'train', 'val',   'test'],
]

split_cropland_fraction_targets = {
    'train': TRAIN_CROPLAND_FRACTION,
    'val': VAL_CROPLAND_FRACTION,
    'test': TEST_CROPLAND_FRACTION,
}

print("\n" + "="*70)
print("STEP 4: EXTRACT PATCHES (5x5 GRID, CROPLAND-AWARE SPLITS)")
print("="*70)
print("\nGrid pattern:")
for i, row in enumerate(grid_pattern):
    print(f"  Row {i}: " + "  ".join([s.upper()[0] for s in row]))
print()

splits = {
    'train': {'patches': [], 'labels': []},
    'val': {'patches': [], 'labels': []},
    'test': {'patches': [], 'labels': []}
}

for i in range(5):
    for j in range(5):
        split = grid_pattern[i][j]
        current = sum(len(p) for p in splits[split]['patches'])

        if current >= TARGET_PATCHES[split]:
            print(f"  Cell [{i},{j}] -> {split:5s}: SKIP (target reached)")
            continue

        remaining = TARGET_PATCHES[split] - current
        needed = min(remaining, MAX_PER_CELL)

        region = {
            'row_start': i * GRID_SIZE,
            'row_end': (i + 1) * GRID_SIZE,
            'col_start': j * GRID_SIZE,
            'col_end': (j + 1) * GRID_SIZE
        }

        current_labels = np.concatenate(splits[split]['labels']) if splits[split]['labels'] else np.array([], dtype=np.uint8)
        current_cropland = int(np.isin(current_labels, list(CROPLAND_LABELS)).sum())
        current_non_cropland = int(len(current_labels) - current_cropland)
        target_fraction = split_cropland_fraction_targets[split]
        target_cropland = int(round(TARGET_PATCHES[split] * target_fraction))
        target_non_cropland = TARGET_PATCHES[split] - target_cropland
        needed_cropland = max(0, min(MAX_PER_CELL, target_cropland - current_cropland, needed))
        needed_non_cropland = max(0, min(MAX_PER_CELL, target_non_cropland - current_non_cropland, needed - needed_cropland))

        patches, labels, counts = extract_quota_patches(
            STACKED_FILE, CORINE_ALIGNED, region,
            needed_cropland, needed_non_cropland,
            GLOBAL_STATS, CROPLAND_LABELS, PATCH_SIZE, STRIDE
        )

        if len(patches) > 0:
            splits[split]['patches'].append(patches)
            splits[split]['labels'].append(labels)
            print(f"  Cell [{i},{j}] -> {split:5s}: +{len(patches):6,} patches (cropland={counts['cropland']:4d}, non-cropland={counts['non_cropland']:4d})")
        else:
            print(f"  Cell [{i},{j}] -> {split:5s}: +0 patches")

print("\nConcatenating...")
for split in ['train', 'val', 'test']:
    if splits[split]['patches']:
        splits[split]['patches'] = np.concatenate(splits[split]['patches'])
        splits[split]['labels'] = np.concatenate(splits[split]['labels'])
        print(f"  {split:5s}: {len(splits[split]['labels']):6,} patches")
        split_cropland = int(np.isin(splits[split]['labels'], list(CROPLAND_LABELS)).sum())
        split_non_cropland = int(len(splits[split]['labels']) - split_cropland)
        print(f"           cropland={split_cropland:6,}, non-cropland={split_non_cropland:6,}")

print("\nStep 4 complete")


STEP 4: EXTRACT PATCHES (5Ãƒâ€”5 GRID, CROPLAND-AWARE TRAIN SPLIT)

Grid pattern:
  Row 0: T  V  T  T  V
  Row 1: V  T  T  V  T
  Row 2: T  T  V  T  T
  Row 3: T  V  T  T  V
  Row 4: V  T  T  V  T

  Cell [0,0] Ã¢â€ â€™ train: +0 patches
  Cell [0,1] Ã¢â€ â€™ val  : + 3,000 patches (cropland=   0, non-cropland=3000)
  Cell [0,2] Ã¢â€ â€™ test : + 4,000 patches
  Cell [0,3] Ã¢â€ â€™ train: + 1,307 patches (cropland=1307, non-cropland=   0)
  Cell [0,4] Ã¢â€ â€™ val  : + 4,000 patches (cropland=1000, non-cropland=3000)
  Cell [1,0] Ã¢â€ â€™ val  : + 3,000 patches (cropland=   0, non-cropland=3000)
  Cell [1,1] Ã¢â€ â€™ test : + 4,000 patches
  Cell [1,2] Ã¢â€ â€™ train: + 2,825 patches (cropland=1518, non-cropland=1307)
  Cell [1,3] Ã¢â€ â€™ val  : SKIP (target reached)
  Cell [1,4] Ã¢â€ â€™ test : + 2,000 patches
  Cell [2,0] Ã¢â€ â€™ test : SKIP (target reached)
  Cell [2,1] Ã¢â€ â€™ train: + 4,000 patches (cropland=1175, non-cropland=2825)
  Cell [2,2] Ã¢â€ â€™ val  : SKIP (target re

## Step 5: Save Dataset

In [7]:
print("\n" + "="*70)
print("STEP 5: SAVE DATASET")
print("="*70)

# Save patches and labels
for split in ['train', 'val', 'test']:
    patches = splits[split]['patches']
    labels = splits[split]['labels']
    
    # Save
    patch_file = Path(OUTPUT_BASE) / f'patches_{split}.npz'
    label_file = Path(OUTPUT_BASE) / f'labels_{split}.npy'
    
    np.savez_compressed(patch_file, patches=patches)
    np.save(label_file, labels)
    
    print(f"\n{split}:")
    print(f"  Shape: {patches.shape}")
    print(f"  Labels: {len(labels):,}")
    print(f"  Size: {patches.nbytes / 1e6:.1f} MB")
    print(f"  Range: [{patches.min():.3f}, {patches.max():.3f}]")

# Save metadata
metadata = {
    'tile': TILE_ID,
    'region': 'Po Valley, Italy',
    'acquisitions': ALL_SAFE_FOLDERS,
    'bands': BANDS,
    'n_bands_total': len(BANDS) * len(ALL_SAFE_FOLDERS),
    'patch_size': PATCH_SIZE,
    'stride': STRIDE,
    'grid': '3x3',
    'grid_cell_size': GRID_SIZE,
    'normalization': 'GLOBAL_percentile_2_98',
    'normalization_note': 'Critical fix: global stats from entire tile, not per-cell',
    'cropland_labels': sorted(CROPLAND_LABELS),
    'train_cropland_fraction_target': TRAIN_CROPLAND_FRACTION,
    'val_cropland_fraction_target': VAL_CROPLAND_FRACTION,
    'test_cropland_fraction_target': TEST_CROPLAND_FRACTION,
    'sampling_note': 'All splits are sampled with explicit cropland/non-cropland quotas so the held-out test set contains enough cropland for stable evaluation.',
    'global_stats': GLOBAL_STATS,
    'splits': {k: int(len(v['labels'])) for k, v in splits.items()},
    'split_class_counts': {
        k: {
            'cropland': int(np.isin(v['labels'], list(CROPLAND_LABELS)).sum()),
            'non_cropland': int(len(v['labels']) - np.isin(v['labels'], list(CROPLAND_LABELS)).sum())
        }
        for k, v in splits.items()
    },
    'created': datetime.now().isoformat()
}

meta_file = Path(OUTPUT_BASE) / 'metadata.json'
with open(meta_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nÃ¢Å“â€œ Metadata saved")

print("\n" + "="*70)
print("Ã¢Å“â€¦ DATASET CREATION COMPLETE!")
print("="*70)
print(f"\nOutput directory: {OUTPUT_BASE}")
print(f"\nFiles created:")
print(f"  - {TILE_ID}_stacked.tif")
print(f"  - corine_aligned_{TILE_ID}.tif")
print(f"  - global_normalization_stats.json  Ã¢â€ Â THE FIX!")
print(f"  - patches_train.npz")
print(f"  - patches_val.npz")
print(f"  - patches_test.npz")
print(f"  - labels_train.npy")
print(f"  - labels_val.npy")
print(f"  - labels_test.npy")
print(f"  - metadata.json")
print(f"\nÃ°Å¸â€Â´ Global normalization applied!")
print(f"Expected accuracy: 30-40% (vs 12% with broken normalization)")
print(f"\nNext: Run the training notebook!")


STEP 5: SAVE DATASET

train:
  Shape: (20000, 5, 5, 16)
  Labels: 20,000
  Size: 32.0 MB
  Range: [0.000, 1.000]

val:
  Shape: (10000, 5, 5, 16)
  Labels: 10,000
  Size: 16.0 MB
  Range: [0.000, 1.000]

test:
  Shape: (10000, 5, 5, 16)
  Labels: 10,000
  Size: 16.0 MB
  Range: [0.000, 1.000]

Ã¢Å“â€œ Metadata saved

Ã¢Å“â€¦ DATASET CREATION COMPLETE!

Output directory: /p/scratch/training2600/teamMozwiloFortune/final_project/data

Files created:
  - T32TMR_stacked.tif
  - corine_aligned_T32TMR.tif
  - global_normalization_stats.json  Ã¢â€ Â THE FIX!
  - patches_train.npz
  - patches_val.npz
  - patches_test.npz
  - labels_train.npy
  - labels_val.npy
  - labels_test.npy
  - metadata.json

Ã°Å¸â€Â´ Global normalization applied!
Expected accuracy: 30-40% (vs 12% with broken normalization)

Next: Run the training notebook!


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"
sns.set_theme(style='whitegrid')

figures_dir = Path(OUTPUT_BASE) / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

with rasterio.open(STACKED_FILE) as src:
    rgb_raw = src.read([3, 2, 1]).astype(np.float32)

rgb_disp = np.zeros_like(rgb_raw, dtype=np.float32)
for src_band_idx, rgb_idx in zip([2, 1, 0], [0, 1, 2]):
    p2 = GLOBAL_STATS[src_band_idx]['p2']
    p98 = GLOBAL_STATS[src_band_idx]['p98']
    rgb_disp[rgb_idx] = np.clip((rgb_raw[rgb_idx] - p2) / (p98 - p2 + 1e-6), 0, 1)

rgb_disp = np.transpose(rgb_disp, (1, 2, 0))
preview = rgb_disp[::10, ::10, :]

split_summary = []
for split in ['train', 'val', 'test']:
    labels = splits[split]['labels']
    cropland = int(np.isin(labels, list(CROPLAND_LABELS)).sum())
    split_summary.append({'Split': split.title(), 'Class': 'Cropland', 'Count': cropland})
    split_summary.append({'Split': split.title(), 'Class': 'Non-cropland', 'Count': int(len(labels) - cropland)})
split_df = pd.DataFrame(split_summary)

sample_rows = []
for split in ['train', 'val', 'test']:
    labels = splits[split]['labels']
    sample_rows.append({
        'Split': split.title(),
        'Total_Patches': int(len(labels)),
        'Cropland': int(np.isin(labels, list(CROPLAND_LABELS)).sum()),
        'Non_Cropland': int(len(labels) - np.isin(labels, list(CROPLAND_LABELS)).sum())
    })
summary_df = pd.DataFrame(sample_rows)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[1.4, 1.0], height_ratios=[1.0, 0.9])
ax_map = fig.add_subplot(gs[:, 0])
ax_bar = fig.add_subplot(gs[0, 1])
ax_table = fig.add_subplot(gs[1, 1])

ax_map.imshow(preview)
ax_map.set_title('Study Area Overview (Sentinel-2 RGB, March acquisition)', fontsize=13)
ax_map.set_xlabel('Column (downsampled pixels)')
ax_map.set_ylabel('Row (downsampled pixels)')

palette = {'Cropland': '#4C78A8', 'Non-cropland': '#F58518'}
sns.barplot(data=split_df, x='Split', y='Count', hue='Class', palette=palette, ax=ax_bar)
ax_bar.set_title('Class Distribution by Dataset Split', fontsize=13)
ax_bar.set_xlabel('Dataset Split')
ax_bar.set_ylabel('Patch Count')
ax_bar.legend(title='Class', frameon=True)

ax_table.axis('off')
summary_table = ax_table.table(
    cellText=summary_df.values,
    colLabels=['Split', 'Total Patches', 'Cropland', 'Non-cropland'],
    cellLoc='center',
    loc='center'
)
summary_table.auto_set_font_size(False)
summary_table.set_fontsize(10)
summary_table.scale(1.15, 1.5)
ax_table.set_title('Dataset Summary Table', fontsize=13, pad=12)

fig.suptitle('Data Overview for the Po Valley Cropland Classification Study', fontsize=15, y=0.97)
fig.text(
    0.02, 0.02,
    'Figure 1. Data overview for the Po Valley study area (tile T32TMR). Left: Sentinel-2 RGB overview of the tile. Top right: cropland versus non-cropland patch counts by split. Bottom right: summary table of patch counts used for training, validation, and testing.',
    ha='left', va='bottom', fontsize=10
)
fig.tight_layout(rect=[0, 0.05, 1, 0.95])
fig.savefig(figures_dir / 'figure1_data_overview.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved figure: figure1_data_overview.png')
